In [ ]:
# Cell 1 - Install
%pip install "git+https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git"

In [ ]:
# Cell 2 - Import
import pandas as pd

from data_processing.ingestion import load_csv
from data_processing.cleaning import (
    remove_duplicates,
    handle_missing_values,
    standardize_dates,
)

print('Package installed and imported successfully')

In [ ]:
# Cell 3 - Load
df = load_csv('/lakehouse/default/Files/bronze/circulation_data.csv')

print(f'Bronze: {len(df)} rows')
print(df.head())

In [ ]:
# Cell 4 - Clean
df = remove_duplicates(df, subset=['transaction_id'])
df = handle_missing_values(df, strategy='drop')
df = standardize_dates(df, ['checkout_date', 'return_date'])

# Force real datetime dtype, regardless of how standardize_dates() formatted these
df['checkout_date'] = pd.to_datetime(df['checkout_date'])
df['return_date'] = pd.to_datetime(df['return_date'])

print(f'Silver: {len(df)} rows')

In [ ]:
# Cell 5 - Validate
assert len(df) > 0, 'no rows left after cleaning'
assert df['transaction_id'].is_unique, 'transaction_id should be unique'
assert df['member_id'].isnull().sum() == 0, 'member_id should have no nulls'
assert df['branch_id'].isnull().sum() == 0, 'branch_id should have no nulls'
assert pd.api.types.is_datetime64_any_dtype(df['checkout_date']), 'checkout_date should be a date'

print('All validation checks passed')
print(df.dtypes)

In [ ]:
# Cell 6 - Write
spark.createDataFrame(df).write.mode('overwrite').saveAsTable('silver_circulation')

print('Saved: silver_circulation')